## Flyweight

---

> **In one line.** A flyweight factors an object's full state into a shared, immutable **intrinsic** part — stored exactly once per type — and a unique, context-dependent **extrinsic** part that is never stored but passed in at call time, collapsing the memory cost of $n$ near-identical objects from $O(n)$ down to $O(k)$ with $k \ll n$.

### 1. State and its factorisation

Let $x$ be one of $n$ objects we must keep alive — say one of $100{,}000$ trees in a game world, or one of millions of characters in a text editor. Write $\text{state}(x)$ for its **complete state**: everything needed to fully describe it. The pattern's whole idea is that this state splits cleanly into two disjoint parts,

$$\boxed{\;\text{state}(x) \;=\; \text{intrinsic}(x) \;\cup\; \text{extrinsic}(x)\;}$$

where $\text{intrinsic}(x)$ is the **shared, immutable** part — identical for every object of the same type (the mesh, texture and colour of all Oaks; the glyph, font and size of every `"A"`) — and $\text{extrinsic}(x)$ is the **unique, context-dependent** part that differs per instance (position on screen, scale). The intrinsic part is *stored*; the extrinsic part is *never stored*, only supplied as a parameter when an operation runs.

### 2. Sharing by type

Let $\text{type}(x)$ classify objects by their intrinsic content, and let $k$ be the number of **distinct types** — e.g. $k = 3$ for Oak, Pine, Birch, with $k \ll n$. The defining sharing law is that same-type objects carry *the same* intrinsic state:

$$\text{intrinsic}(x) = \text{intrinsic}(y) \qquad \text{if } \text{type}(x) = \text{type}(y).$$

This is realised not as a value copy but as **physical identity in memory**. Writing $\text{addr}(\cdot)$ for the address of the shared intrinsic object, the law strengthens to

$$\text{addr}(\text{intrinsic}(x)) = \text{addr}(\text{intrinsic}(y)) \qquad \text{if } \text{type}(x) = \text{type}(y),$$

i.e. one object, many references — they literally point to the same object in RAM.

### 3. The pool and the data flow

A **factory** maintains a pool of exactly $k$ flyweight objects, keyed by type. A request for a given type either returns the existing flyweight or, on first sight, creates and stores it — so duplicates can never arise. At call time the extrinsic state $e$ is threaded through the shared flyweight:

$$\text{client} \xrightarrow{\;\text{type}\;} \underbrace{\text{factory}}_{\text{pool of }k} \xrightarrow{\;\text{shared flyweight}\;} \underbrace{\text{op}(\,\text{intrinsic},\, e\,)}_{e\ =\ \text{extrinsic, passed in}}$$

### 4. The memory result

Because intrinsic state is stored once per type and extrinsic state is not stored at all, total memory drops from holding the full state of all $n$ objects to holding only $k$ shared flyweights:

$$\boxed{\;\text{memory: } O(n) \;\longrightarrow\; O(k), \qquad k \ll n\;}$$

### 5. The three conditions

1. **Intrinsic is shared.** Same-type objects point to one object in RAM:
   $$\text{addr}(\text{intrinsic}(x)) = \text{addr}(\text{intrinsic}(y)) \quad \text{for } \text{type}(x) = \text{type}(y).$$
2. **Extrinsic is never stored.** Extrinsic state is passed as a parameter at render/call time; storing it would defeat the memory saving, breaking $O(n) \to O(k)$.
3. **Factory ensures uniqueness.** The factory holds a pool of $k$ flyweights and never creates a duplicate — the same type always returns the same object, which is exactly what enforces condition 1.

> 🌲 Rendering $100{,}000$ trees. Intrinsic — mesh, texture, colour — is the same for all Oaks and is stored once. Extrinsic — position, scale — is unique per tree and passed at render time. Memory drops from $O(100000)$ to $O(3)$: Oak, Pine, Birch.

### Exercise 13 — Character Renderer

---

**Scenario:** A text editor renders millions of characters. Intrinsic: char, font, size (same for all "A"s). Extrinsic: position $x, y$ on screen (unique per character instance).

**Your task:** Build `CharacterFlyweight` (stores intrinsic) and `FlyweightFactory` (returns same object for repeated types). Verify with `is`.

```python
a1 = factory.get("A", "Arial", 12)
a2 = factory.get("A", "Arial", 12)
print(a1 is a2)        # True — addr(intrinsic(a1)) = addr(intrinsic(a2))
a1.render(x=10, y=20)  # extrinsic passed at call time, never stored
```

**Hints**

- The factory stores `self._pool = {}` keyed by `(char, font, size)`. If the key exists → return the same object (satisfying $\text{addr}(\text{intrinsic}(x)) = \text{addr}(\text{intrinsic}(y))$). If not → create and store it.
- `render(x, y)` takes position as parameters — never stores them. This is the extrinsic state: passed in, not retained.

In [ ]:
# --------------------------------
# Flyweight: stores ONLY intrinsic(x) — char, font, size (shared, immutable)

class CharacterFlyweight:
    def __init__(self, char, font, size):
        self.char = char                        # intrinsic(x)
        self.font = font                        # intrinsic(x)
        self.size = size                        # intrinsic(x)

    def render(self, x, y):                      # x, y are extrinsic — passed in, never stored
        print(f"Render '{self.char}' ({self.font} {self.size}) at ({x}, {y})")

# --------------------------------
# Factory: holds a pool of k flyweights — same type always returns the SAME object

class FlyweightFactory:
    def __init__(self):
        self._pool = {}                          # key (char, font, size) -> flyweight

    def get(self, char, font, size):
        # key = (char, font, size)
        # if key in pool -> return same object  (addr(intrinsic(x)) == addr(intrinsic(y)))
        # else -> create CharacterFlyweight, store under key, return it
        ...

# --------------------------------
factory = FlyweightFactory()
a1 = factory.get("A", "Arial", 12)
a2 = factory.get("A", "Arial", 12)
print(a1 is a2)        # expect True — one intrinsic object, two references
a1.render(x=10, y=20)  # extrinsic passed at call time

### Exercise 14 — Forest Tree Renderer

---

**Scenario:** 50,000 trees. Intrinsic: tree type mesh and texture ($k = 3$ types). Extrinsic: position and scale ($n = 50000$ unique values). Goal: $O(n) \rightarrow O(k)$ memory.

**Your task:** Build `TreeType` (flyweight), `Tree` (holds extrinsic + reference to flyweight), and `TreeFactory`. Verify factory count stays at $k$.

```python
forest = [Tree(x, y, factory.get_tree_type("Oak")) for x, y in positions]
print(len(factory._pool))   # stays at k = 3 no matter how many trees
```

**Hints**

- `Tree` is lightweight: stores only $x$, $y$, and a reference to a `TreeType`. `TreeType` stores the heavy intrinsic data. Only $k = 3$ `TreeType` objects ever exist, referenced 50,000 times.
- Measure with `sys.getsizeof`: compare memory with and without Flyweight for 1000 trees to see $O(n) \rightarrow O(k)$ concretely.

In [ ]:
import sys

# --------------------------------
# Flyweight: heavy intrinsic state — shared across all trees of the same type

class TreeType:
    def __init__(self, name, mesh, texture):
        self.name = name                         # intrinsic(x)
        self.mesh = mesh                         # intrinsic(x) — heavy, stored once per type
        self.texture = texture                   # intrinsic(x)

    def render(self, x, y, scale):               # extrinsic passed in, never stored
        print(f"Render {self.name} at ({x}, {y}) scale={scale}")

# --------------------------------
# Context: lightweight — stores extrinsic(x) + a reference to the shared flyweight

class Tree:
    def __init__(self, x, y, tree_type):
        self.x = x                               # extrinsic(x)
        self.y = y                               # extrinsic(x)
        self.tree_type = tree_type               # reference to shared intrinsic

    def render(self, scale=1.0):
        self.tree_type.render(self.x, self.y, scale)

# --------------------------------
# Factory: pool of k TreeType flyweights — same name returns the SAME object

class TreeFactory:
    def __init__(self):
        self._pool = {}                          # name -> TreeType

    def get_tree_type(self, name):
        # if name in pool -> return same TreeType  (k objects, n references)
        # else -> create TreeType (mesh/texture for this name), store, return
        ...

# --------------------------------
factory = TreeFactory()
names = ["Oak", "Pine", "Birch"]
forest = [Tree(i, i * 2, factory.get_tree_type(names[i % 3])) for i in range(1000)]

print("trees (n):", len(forest))            # n = 1000
print("tree types (k):", len(factory._pool)) # expect 3 — stays at k
forest[0].render(scale=1.5)